# Tạo dữ liệu Quick Link từ file .md

Notebook này quét nội dung các file `.md` (dựa vào tên file + heading) để tự sinh
phần `items` (title / slug / children.anchor) cho MỘT (nikaya, edition).

Những gì notebook **không** đoán được — vì đó là thuộc tính của cả bộ kinh/bản dịch,
không nằm trong nội dung 1 file — sẽ để `"???"` cho bạn tự điền:
`folder`, `label`, `path`, `index_length`.

**Quy ước mà notebook giả định** (chỉnh lại ở phần CONFIG nếu khác):
- Tên file (bỏ `.md`) là `slug`, dạng `<nikaya>-<số>-...`, ví dụ `mn-007-the-simile-of-the-cloth`
  → số thứ tự đầu tiên (`top index`) = `7`.
- Dòng `# ...` (H1) đầu tiên trong file là `title` của kinh.
- Các heading cấp `###` (H3, xem `CHILD_HEADING_LEVEL`) có dạng số đầu dòng như
  `### 2.2 Dutiyakassapasutta` được coi là mốc "đoạn con" → sinh `children`.
  Số đầu tiên trong heading con phải khớp với top index của file (nếu không sẽ có warning),
  các số còn lại tạo thành đường dẫn lồng nhau trong `children`.

**Về anchor:** notebook tự đoán anchor bằng một hàm slugify phổ biến (lowercase, thay
ký tự không phải chữ/số bằng `-`), khớp đúng với ví dụ `2-2-dutiyakassapasutta` bạn đưa.
Tuy nhiên đây là suy đoán — VitePress dùng `markdown-it-anchor` nội bộ và có thể xử lý
dấu Pali/Việt (ā, ṁ, ñ, ...) khác đi. **Nên đối chiếu lại với anchor thật** (mở trang đã
build, bấm vào link cạnh heading, xem `#...` trên thanh địa chỉ) trước khi dùng chính thức,
đặc biệt với các tiêu đề có dấu.

## 1. CONFIG — sửa các giá trị dưới đây

In [ ]:
import os, re, json

# Thư mục chứa các file .md
SOURCE_DIR = "."

# Danh sách tên file .md cần xử lý (chỉ tên file, không cần đường dẫn đầy đủ)
FILES = [
    "mn-007-the-simile-of-the-cloth.md",
    "sn-002-devaputtasamyutta.md",
]

# Heading cấp mấy được coi là "đoạn con" (### = 3, ## = 2, ...)
CHILD_HEADING_LEVEL = 3


## 2. Các hàm xử lý

In [ ]:
TOP_INDEX_RE = re.compile(r'^[a-z]+-0*(\d+)')
H1_RE = re.compile(r'^#\s+(.*)$', re.MULTILINE)
CHILD_RE = re.compile(r'^#{%d}\s+(.*)$' % CHILD_HEADING_LEVEL)
NUM_PREFIX_RE = re.compile(r'^(\d+(?:\.\d+)+)\.?\s*')


def slugify(text):
    """Đoán anchor kiểu markdown-it-anchor. Xem ghi chú ở đầu notebook."""
    text = text.strip()
    text = re.sub(r'`([^`]*)`', r'\1', text)      # bỏ backtick code
    text = text.lower()
    text = re.sub(r'[^\w\s-]', '-', text, flags=re.UNICODE)
    text = re.sub(r'[\s_]+', '-', text)
    text = re.sub(r'-+', '-', text)
    return text.strip('-')


def top_index_from_slug(slug):
    m = TOP_INDEX_RE.match(slug.lower())
    return m.group(1) if m else None


def extract_h1(text):
    m = H1_RE.search(text)
    return m.group(1).strip() if m else None


def insert_child(item, path, anchor):
    node = item
    for i, key in enumerate(path):
        node.setdefault("children", {})
        node["children"].setdefault(key, {})
        node = node["children"][key]
        if i == len(path) - 1:
            node["anchor"] = anchor


def process_file(dirpath, filename, items, warnings):
    slug = filename[:-3] if filename.endswith(".md") else filename
    filepath = os.path.join(dirpath, filename)
    if not os.path.isfile(filepath):
        warnings.append(f"{filename}: không tìm thấy file, bỏ qua")
        return

    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    top_index = top_index_from_slug(slug)
    title = extract_h1(text)

    item = {}
    item["title"] = title if title else "??? (không tìm thấy H1)"
    if not title:
        warnings.append(f"{filename}: không tìm thấy dòng H1 (# ...)")
    item["slug"] = slug

    for line in text.splitlines():
        m = CHILD_RE.match(line)
        if not m:
            continue
        heading_text = m.group(1).strip()
        num_match = NUM_PREFIX_RE.match(heading_text)
        if not num_match:
            continue  # heading cấp con nhưng không có số đầu dòng -> bỏ qua
        numbers = num_match.group(1).split(".")
        if top_index and numbers[0] != str(int(top_index)):
            warnings.append(
                f'{filename}: heading "{heading_text}" có số đầu ({numbers[0]}) '
                f'khác top index suy từ tên file ({top_index}) — kiểm tra lại'
            )
        remaining = numbers[1:]
        if not remaining:
            continue
        anchor = slugify(heading_text)
        insert_child(item, remaining, anchor)

    if top_index is None:
        warnings.append(f"{filename}: không suy ra được số thứ tự từ tên file, cần điền tay")
        key = f"???({slug})"
    else:
        key = str(int(top_index))
        if key in items:
            warnings.append(f'{filename}: trùng key "{key}" với 1 file khác — kiểm tra lại')

    items[key] = item


## 3. Chạy xử lý

In [ ]:
items = {}
warnings = []

for fn in FILES:
    process_file(SOURCE_DIR, fn, items, warnings)

if warnings:
    print("⚠️  Cảnh báo:")
    for w in warnings:
        print(" -", w)
else:
    print("Không có cảnh báo.")


## 4. Kết quả — dán vào `quicklink-data.js`

Các trường `"???"` là chỗ bạn tự điền (`folder`, edition key, `label`, `path`, `index_length`).

In [ ]:
output = {
    "folder": "???",
    "editions": {
        "???edition_key???": {
            "label": "???",
            "path": "???",
            "index_length": "???",
            "items": items,
        }
    },
}

print(json.dumps(output, ensure_ascii=False, indent=2))


## 5. (Tùy chọn) Ghi ra file JSON

Chạy cell dưới nếu muốn lưu kết quả ra file thay vì chỉ copy từ output ở trên.

In [ ]:
OUT_PATH = "quicklink-data.generated.json"

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Đã ghi: {OUT_PATH}")
